# 合成数据生成器 —— Google Colab 版

用 **Hugging Face** 开源 Instruct 模型 + **Gradio** 界面，按模板（或自定义 prompt）生成**结构化 JSON** 合成数据，并用 JSON Schema 做校验。

## 练习目标（理念）

- 模板里写清 `schema` + `prompt_template`（含 `{{count}}` / `{{tone}}` / `{{schema}}` 占位符）
- 本地加载 Causal LM，把模型原文尽量解析成 JSON
- Gradio 里调温度、max tokens，并可查看 raw 输出便于排错

## 和本课第 3 周的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 合成数据 / 结构化输出 | 模板 + `safe_json_load` |
| Hugging Face Transformers | `AutoTokenizer` / `AutoModelForCausalLM` |
| Schema 校验 | `jsonschema.validate` |
| Gradio 产品化 UI | `gr.Blocks` + 生成按钮 |

## 怎么跑（Colab）

1. 在 [Hugging Face Tokens](https://huggingface.co/settings/tokens) 创建**读权限** token，放进 Colab Secrets，名称为 `HF_TOKEN`
2. 从上到下依次运行单元格（依赖 → 导入 → 密钥 → 类定义 → 初始化 → 启动 Gradio）
3. 在界面选模板或切到 `custom`，点 Generate 查看 JSON / raw / validation


## 1. 安装依赖项


In [ ]:
# ========== 安装运行所需包（Colab shell 魔法） ==========
# gradio：UI；httpx：HTTP；python-dotenv：.env；jsonschema：校验；
# transformers / torch / huggingface-hub：加载与推理开源模型
!pip install gradio>=4.0.0 httpx>=0.24.0 python-dotenv>=1.0.0 jsonschema>=4.19.0 transformers>=4.30.0 torch>=2.0.0 huggingface-hub>=0.16.0



## 2. 导入库和设置


In [ ]:
# ========== 导入标准库、类型标注、HF/Torch、jsonschema、Gradio ==========

# os：读环境变量（备用 HF_TOKEN）
import os
# json：序列化 schema / 美化输出
import json
# re：从模型杂文里抠 JSON 数组/对象
import re
# 类型标注：Any / Dict / List / Optional / Tuple
from typing import Any, Dict, List, Optional, Tuple
# Path：路径工具（本笔记本后续可扩展读写）
from pathlib import Path

# PyTorch：设备与半精度
import torch
# 分词器 + 因果语言模型
from transformers import AutoTokenizer, AutoModelForCausalLM
# Hugging Face Hub 登录
from huggingface_hub import login
# JSON Schema 校验与校验错误类型
from jsonschema import validate as jsonschema_validate, ValidationError
# Gradio：搭生成器界面
import gradio as gr
# Colab Secrets：取 HF_TOKEN
from google.colab import userdata

print("Libraries imported successfully!")


## 3. API 密钥配置


In [ ]:
# ========== 从 Colab Secrets 读取 Hugging Face Token ==========

# userdata 里的变量名必须是 HF_TOKEN（与 Secrets 名称一致）
HF_TOKEN = userdata.get('HF_TOKEN') # use the name of the variable in the userdata

print("API key configured successfully!")


## 4. 实用函数


In [ ]:
# ========== safe_json_load：从模型原文里尽量抠出合法 JSON ==========

def safe_json_load(text: str) -> Optional[Any]:
    """
    Attempt to parse JSON. Heuristics:
      1) direct json.loads
      2) extract first [...] block
      3) extract first {...} block
      4) try to find JSON after common prefixes
    Returns parsed object or None.
    """
    # 去掉首尾空白，减少无关失败
    text = text.strip()
    
    # 调试：打印长度与前 200 字符，方便对照模型是否加了废话前缀
    print(f"DEBUG: Attempting to parse JSON from text (length: {len(text)})")
    print(f"DEBUG: First 200 chars: {text[:200]}")
    
    # 路径 1：整段就是合法 JSON
    try:
        result = json.loads(text)
        print(f"DEBUG: Direct JSON parse successful")
        return result
    except Exception as e:
        print(f"DEBUG: Direct JSON parse failed: {e}")

    # 路径 2：用正则抓第一个 [...]（合成数据多为 JSON 数组）
    arr_match = re.search(r'(\[.*?\])', text, flags=re.S)
    if arr_match:
        try:
            result = json.loads(arr_match.group(1))
            print(f"DEBUG: Array regex match successful")
            return result
        except Exception as e:
            print(f"DEBUG: Array regex match failed: {e}")

    # 路径 3：抓第一个 {...} 对象
    obj_match = re.search(r'(\{.*?\})', text, flags=re.S)
    if obj_match:
        try:
            result = json.loads(obj_match.group(1))
            print(f"DEBUG: Object regex match successful")
            return result
        except Exception as e:
            print(f"DEBUG: Object regex match failed: {e}")

    # 路径 4：常见英文前缀 / markdown 代码围栏后再抓数组或对象（pattern 字符串勿改）
    json_patterns = [
        r'(?:Here is the JSON:|```json|JSON:|Output:)\s*(\[.*?\])',
        r'(?:Here is the JSON:|```json|JSON:|Output:)\s*(\{.*?\})',
    ]
    
    for pattern in json_patterns:
        match = re.search(pattern, text, flags=re.S | re.I)
        if match:
            try:
                result = json.loads(match.group(1))
                print(f"DEBUG: Pattern match successful")
                return result
            except Exception as e:
                print(f"DEBUG: Pattern match failed: {e}")
                continue

    # 全部失败：返回 None，上层会标 parse_error
    print(f"DEBUG: All parsing attempts failed")
    return None

print("Utility functions defined successfully!")


## 5. 模板数据


In [ ]:
# ========== TEMPLATES：每种合成任务的 schema + 英文 prompt_template ==========
# prompt_template / schema 字段会影响模型行为与校验——保持英文与结构原样

TEMPLATES = {
    # 用户画像：姓名、邮箱、年龄等
    "user_profile": {
        "id": "user_profile",
        "schema": {
            "type": "object",
            "properties": {
                "first_name": { "type": "string" },
                "last_name": { "type": "string" },
                "email": { "type": "string", "format": "email" },
                "age": { "type": "integer", "minimum": 18, "maximum": 80 },
                "country": { "type": "string" }
            },
            "required": ["first_name", "last_name", "email", "age"]
        },
        "prompt_template": "Generate {{count}} user profile(s) as a JSON array matching this schema: {{schema}}. Tone: {{tone}}. Output ONLY the JSON array, no other text."
    },
    # 职位描述：标题、公司、地点、要求与福利
    "job_description": {
        "id": "job_description",
        "schema": {
            "type": "object",
            "properties": {
                "title": { "type": "string" },
                "company": { "type": "string" },
                "location": { "type": "string" },
                "salary_range": { "type": "string" },
                "requirements": { "type": "array", "items": { "type": "string" } },
                "benefits": { "type": "array", "items": { "type": "string" } }
            },
            "required": ["title", "company", "location"]
        },
        "prompt_template": "Generate {{count}} job description(s) as a JSON array matching this schema: {{schema}}. Tone: {{tone}}. Output ONLY the JSON array, no other text."
    },
    # 产品规格：名称、品类、价格、特性等
    "product_spec": {
        "id": "product_spec",
        "schema": {
            "type": "object",
            "properties": {
                "name": { "type": "string" },
                "category": { "type": "string" },
                "price": { "type": "number" },
                "description": { "type": "string" },
                "features": { "type": "array", "items": { "type": "string" } },
                "in_stock": { "type": "boolean" }
            },
            "required": ["name", "category", "price"]
        },
        "prompt_template": "Generate {{count}} product specification(s) as a JSON array matching this schema: {{schema}}. Tone: {{tone}}. Output ONLY the JSON array, no other text."
    },
    # 地址：街道、城市、州、邮编等
    "address": {
        "id": "address",
        "schema": {
            "type": "object",
            "properties": {
                "street": { "type": "string" },
                "city": { "type": "string" },
                "state": { "type": "string" },
                "zip_code": { "type": "string" },
                "country": { "type": "string" }
            },
            "required": ["street", "city", "state", "zip_code"]
        },
        "prompt_template": "Generate {{count}} address(es) as a JSON array matching this schema: {{schema}}. Tone: {{tone}}. Output ONLY the JSON array, no other text."
    }
}

print("Template data loaded successfully!")
print(f"Available templates: {list(TEMPLATES.keys())}")


## 6. 核心类


In [ ]:
# ========== TemplateRegistry + PromptEngine：查模板并渲染占位符 ==========

class TemplateRegistry:
    # 用字典包一层，方便以后换成文件/数据库存储而不改调用方
    def __init__(self, templates: Dict[str, Any]):
        self.templates = templates

    def get_ids(self) -> List[str]:
        # 返回所有模板 id，供 Gradio 下拉框使用
        return list(self.templates.keys())

    def get_template(self, template_id: str) -> Optional[Dict[str, Any]]:
        # 按 id 取完整模板；没有则 None
        return self.templates.get(template_id)

class PromptEngine:
    def build_prompt(self, template: Dict[str, Any], params: Dict[str, Any]) -> str:
        """
        Build a single prompt string.
        Template must include 'prompt_template' and 'schema'.
        params may include count, tone, etc.
        """
        # 取出模板字符串与 schema（缺省为空）
        tpl = template.get("prompt_template", "")
        schema = template.get("schema", {})
        # 渲染最小占位符：{{count}}、{{tone}}、{{schema}}
        prompt = tpl.replace("{{count}}", str(params.get("count", 1)))
        prompt = prompt.replace("{{tone}}", str(params.get("tone", "concise")))
        # schema 以 JSON 字符串嵌入 prompt，让模型「照着字段生成」
        prompt = prompt.replace("{{schema}}", json.dumps(schema))
        return prompt

print("TemplateRegistry and PromptEngine classes defined!")


In [ ]:
# ========== HFClient：登录 Hub、加载模型、generate 出文本 ==========

class HFClient:
    def __init__(self, model_id: Optional[str] = None, api_key: Optional[str] = None):
        # 默认模型 id（可在初始化时覆盖）；字符串勿改
        self.model_id = model_id or "meta-llama/Meta-Llama-3.1-8B-Instruct"
        # 密钥优先参数，否则读环境变量 HF_TOKEN
        self.api_key = api_key or os.getenv("HF_TOKEN")
        
        # 没有密钥直接失败（错误文案保持英文原样）
        if not self.api_key:
            raise RuntimeError("HF_TOKEN not provided in environment.")
        
        print(f"DEBUG: Initializing HFClient with model: {self.model_id}")
        
        # 登录 Hugging Face Hub（gated 模型需要）
        try:
            login(token=self.api_key, add_to_git_credential=True)
            print("DEBUG: Successfully logged in to Hugging Face")
        except Exception as e:
            print(f"DEBUG: Login failed: {e}")
            raise RuntimeError(f"Failed to login to Hugging Face: {e}")
        
        # 加载分词器与因果语言模型
        try:
            print("DEBUG: Loading tokenizer...")
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_id)
            print("DEBUG: Loading model...")
            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_id,
                dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
                device_map="auto" if torch.cuda.is_available() else None
            )
            print("DEBUG: Model loaded successfully")
        except Exception as e:
            print(f"DEBUG: Model loading failed: {e}")
            raise RuntimeError(f"Failed to load model {self.model_id}: {e}")

    def generate(self, prompt: str, temperature: float = 0.2, max_tokens: int = 512) -> Tuple[str, Dict[str, Any]]:
        # 记录 prompt 长度，便于排查超长输入
        print(f"DEBUG: Generating text with prompt length: {len(prompt)}")
        
        try:
            # 文本 → token 张量
            inputs = self.tokenizer(prompt, return_tensors="pt")
            
            # 有 CUDA 时把每个张量搬到 GPU，与模型设备一致
            if torch.cuda.is_available():
                inputs = {k: v.cuda() for k, v in inputs.items()}
            
            # 无梯度生成；pad_token_id 设为 eos，避免部分模型警告
            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=max_tokens,
                    temperature=temperature,
                    do_sample=True,
                    pad_token_id=self.tokenizer.eos_token_id
                )
            
            # 整段解码（可能仍含 prompt 前缀）
            generated_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            
            # 若解码结果以 prompt 开头，则剥掉 prompt，只留续写部分
            if generated_text.startswith(prompt):
                generated_text = generated_text[len(prompt):].strip()
            
            print(f"DEBUG: Generated text length: {len(generated_text)}")
            print(f"DEBUG: Generated text preview: {generated_text[:200]}")
            
            return generated_text, {"status": "success", "model": self.model_id}
            
        except Exception as e:
            # 失败时仍返回字符串 + error meta（文案保持英文）
            print(f"DEBUG: Generation failed: {e}")
            return f"Generation error: {str(e)}", {"status": "error", "error": str(e)}

print("HFClient class defined!")


In [ ]:
# ========== Validator（schema 校验）+ Generator（编排生成流水线） ==========

class Validator:
    def validate(self, schema: Dict[str, Any], data: Any) -> Dict[str, Any]:
        # 空 schema：不做约束，直接视为合法
        if not schema:
            return {"valid": True, "errors": []}
        try:
            # 若顶层是 list：对每个元素分别按 object schema 校验
            if isinstance(data, list):
                for item in data:
                    jsonschema_validate(instance=item, schema=schema)
            else:
                jsonschema_validate(instance=data, schema=schema)
            return {"valid": True, "errors": []}
        except ValidationError as e:
            return {"valid": False, "errors": [str(e)]}
        except Exception as e:
            return {"valid": False, "errors": [str(e)]}

class Generator:
    def __init__(self, hf_client: HFClient, registry: TemplateRegistry, prompt_engine: PromptEngine, validator: Validator):
        # 依赖注入：模型客户端、模板库、提示引擎、校验器
        self.hf = hf_client
        self.registry = registry
        self.engine = prompt_engine
        self.validator = validator

    def generate(self,
                 template_id: Optional[str],
                 params: Dict[str, Any],
                 custom_prompt: Optional[str] = None,
                 temperature: float = 0.2,
                 max_tokens: int = 512) -> Dict[str, Any]:
        # 自定义 prompt：造一个无 schema 的临时模板；否则从 registry 取
        if custom_prompt:
            template = {"id": "custom", "prompt_template": custom_prompt, "schema": {}}
        else:
            template = self.registry.get_template(template_id) or {"id": template_id, "prompt_template": "", "schema": {}}

        # 渲染占位符得到最终发给模型的字符串
        prompt = self.engine.build_prompt(template, params)
        try:
            raw_text, meta = self.hf.generate(prompt, temperature=temperature, max_tokens=max_tokens)
        except Exception as exc:
            return {"status": "error", "output": None, "raw_model_text": "", "validation": {"valid": False, "errors": [str(exc)]}}

        # 尽力解析 JSON；失败则带回 raw 文本并标 parse_error
        parsed = safe_json_load(raw_text)
        if parsed is None:
            return {"status": "error", "output": None, "raw_model_text": raw_text, "validation": {"valid": False, "errors": ["parse_error"]}}

        # 用模板 schema 校验解析结果
        validation = self.validator.validate(template.get("schema", {}), parsed)
        return {"status": "ok", "output": parsed, "raw_model_text": raw_text, "validation": validation}

print("Validator and Generator classes defined!")


## 7. 初始化组件


In [ ]:
# ========== 组装流水线：Registry → PromptEngine → HFClient → Validator → Generator ==========

# 用上面定义的 TEMPLATES 建注册表
registry = TemplateRegistry(TEMPLATES)
# 占位符渲染器（无状态）
prompt_engine = PromptEngine()
# 指定 Llama 3.1 8B Instruct，并把 Colab 读到的 HF_TOKEN 传进去
hf_client = HFClient(model_id="meta-llama/Meta-Llama-3.1-8B-Instruct", api_key=HF_TOKEN)
# JSON Schema 校验器
validator = Validator()
# 把四者绑成统一 generate() 入口
generator = Generator(hf_client, registry, prompt_engine, validator)

print("All components initialized successfully!")
print(f"Available templates: {registry.get_ids()}")


## 8. Gradio 界面


In [ ]:
# ========== Gradio 回调：把 UI 参数转成 Generator.generate 结果三元组 ==========

def generate_action(template_id: str, count: int, tone: str, temperature: float, max_tokens: int, custom_prompt: str, show_raw: bool):
    # 模板占位符用到的参数
    params = {"count": count, "tone": tone}
    # 选 custom：走自定义 prompt 分支；否则按模板 id 生成
    if template_id == "custom":
        result = generator.generate(None, params, custom_prompt=custom_prompt, temperature=temperature, max_tokens=max_tokens)
    else:
        result = generator.generate(template_id, params, custom_prompt=None, temperature=temperature, max_tokens=max_tokens)
    # 拆出结构化输出、原文、校验结果
    output = result.get("output")
    raw = result.get("raw_model_text", "")
    validation = result.get("validation", {"valid": False, "errors": []})
    # 有解析结果才 pretty-print JSON，否则空字符串
    json_text = json.dumps(output, indent=2) if output is not None else ""
    
    # 未勾选 show_raw 时，第二路输出用这段 debug 摘要代替全文
    debug_info = f"Status: {result.get('status', 'unknown')}\nRaw length: {len(raw)} chars\nFirst 200 chars: {raw[:200]}"
    
    if show_raw:
        return json_text, raw, json.dumps(validation, indent=2)
    return json_text, debug_info, json.dumps(validation, indent=2)

print("Generate function defined!")


In [ ]:
# ========== 构建 Gradio Blocks：左栏参数、右栏 JSON/raw/validation ==========

# 下拉选项 = 模板 id + 额外的 custom
templates = registry.get_ids()
templates_dropdown = templates + ["custom"]

with gr.Blocks(title="Synthetic Data Generator - Colab") as demo:
    with gr.Row():
        with gr.Column(scale=1):
            # 模板选择；默认第一项
            template = gr.Dropdown(label="Template", choices=templates_dropdown, value=templates_dropdown[0])
            # 生成条数 1–10
            count = gr.Slider(label="Count", minimum=1, maximum=10, value=1, step=1)
            # 语气占位符，默认 concise
            tone = gr.Textbox(label="Tone (optional)", value="concise")
            # 采样温度
            temperature = gr.Slider(label="Temperature", minimum=0.0, maximum=1.0, value=0.2, step=0.05)
            # 最大新 token 数
            max_tokens = gr.Number(label="Max tokens", value=512)
            # 仅当 Template=custom 时显示（初始 hidden）
            custom_prompt = gr.Textbox(label="Custom prompt (used if Template=custom)", lines=6, visible=False)
            # 是否在第二路展示完整 raw
            show_raw = gr.Checkbox(label="Show raw model output", value=False)
            gen_btn = gr.Button("Generate")
        with gr.Column(scale=1):
            # 结构化 JSON 输出
            output = gr.Code(label="Output (JSON)", language="json")
            # raw 或 debug 摘要
            raw_out = gr.Textbox(label="Raw model output", lines=8)
            # 校验结果 JSON
            validation = gr.Code(label="Validation", language="json")

    def on_template_change(t):
        # 选中 custom 时显示自定义 prompt 文本框
        return gr.update(visible=(t == "custom"))

    template.change(on_template_change, inputs=[template], outputs=[custom_prompt])

    # 点击 Generate → generate_action → 三路输出
    gen_btn.click(fn=generate_action,
                  inputs=[template, count, tone, temperature, max_tokens, custom_prompt, show_raw],
                  outputs=[output, raw_out, validation])

print("Gradio interface created!")


## 9. 启动界面


In [ ]:
# ========== 启动 Gradio：share 外链 + debug 详细报错 ==========

demo.launch(share=True, debug=True)


## 使用说明

1. **选择模板**：从预定义模板（`user_profile` / `job_description` / `product_spec` / `address`）里选，或选 `custom` 写自己的提示
2. **配置参数**：
   - **Count**：生成条数（1–10），填进 `{{count}}`
   - **Tone**：生成语气（如 `concise` / detailed / professional），填进 `{{tone}}`
   - **Temperature**：随机性（0.0 更稳，1.0 更跳）
   - **Max tokens**：续写最大长度
3. **自定义提示**：Template=`custom` 时，在文本框写英文 prompt（示例见下；发给模型的英文保持原样）
4. **生成**：点 **Generate**
5. **查看结果**：
   - **Output (JSON)**：解析后的结构化数据
   - **Raw model output**：模型原文（或 debug 摘要）
   - **Validation**：是否通过模板 schema

## 自定义提示示例（英文，可直接粘贴）

**生成产品评论：**
```
Generate 3 product reviews as a JSON array. Each review should have: rating (1-5), title, content, reviewer_name, date. Tone: honest and detailed.
```

**生成社交媒体帖子：**
```
Generate 2 social media posts as a JSON array. Each post should have: platform, content, hashtags (array), engagement_metrics (likes, shares, comments). Tone: engaging and modern.
```

**生成客户支持工单：**
```
Generate 3 customer support tickets as a JSON array. Each ticket should have: ticket_id, customer_email, subject, description, priority (low/medium/high), status. Tone: realistic and varied.
```
